# MIMIC-IV readmission cohort

Predict observed unplanned hospital readmission 30 days after discharge using
features available by 24 hours after admission. Select the HRP `.venv`
kernel and run from `HRP` or `HRP/src`. The default mode inspects schema and
estimates query bytes. Set `RUN_COHORT = True` to extract and save a new
private run. To review an existing run offline, set `EXISTING_RUN` to its
absolute path. Clear cell outputs before committing.

In [1]:
import os
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display
from google.cloud import bigquery

SOURCE = Path.cwd() / "src" if (Path.cwd() / "src" / "run_cohort.py").is_file() else Path.cwd()
if not (SOURCE / "run_cohort.py").is_file():
    raise RuntimeError("Open the notebook with HRP or HRP/src as the working directory.")
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))

from run_cohort import (DEFAULT_CAP, DEFAULT_DIRECTORY, DEFAULT_PROJECT,
                        inspect_queries, inspect_schema, run_workflow, save_run)

PROJECT_ID = os.environ.get("MIMIC_BQ_PROJECT", DEFAULT_PROJECT)
OUTPUT_DIR = Path(os.environ.get("MIMIC_OUTPUT_DIR", DEFAULT_DIRECTORY))
RUN_COHORT = False
EXISTING_RUN = None  # e.g. Path("/absolute/path/to/run-...")
MAX_BYTES_BILLED = DEFAULT_CAP

## Source schema and scan estimates

`patients` provides anchor age/year, gender and date of death.
`admissions` provides encounter identifiers, times, death flag and
administrative admission categories. `labevents` provides numeric results,
units and chart/result times. `d_labitems` defines lab analytes, fluids and
categories. The metadata check verifies fields and types against
`physionet-data.mimiciv_3_1_hosp`. The dry runs scan zero data and estimate
bytes for each column-restricted query.

In [2]:
output = Path(EXISTING_RUN).expanduser() if EXISTING_RUN is not None else None
if output is None:
    client = bigquery.Client(project=PROJECT_ID)
    location, schema = inspect_schema(client)
    plans = inspect_queries(client, location)
    display(schema)
    display(pd.DataFrame([
        {"query": name, "estimated_bytes": size} for name, (_, _, size) in plans.items()
    ]))
else:
    display(pd.read_csv(output / "aggregates/source_schema.csv"))

/Users/sghosh/Documents/Github/AI380H/HRP/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


,table,source_rows,used_fields
0,patients,364627,"subject_id INTEGER, gender STRING, anchor_age ..."
1,admissions,546028,"subject_id INTEGER, hadm_id INTEGER, admittime..."
2,d_labitems,1650,"itemid INTEGER, label STRING, fluid STRING, ca..."
3,labevents,158374764,"labevent_id INTEGER, subject_id INTEGER, hadm_..."


,query,estimated_bytes
0,patients,10151337
1,cohort,50488454
2,lab_itemids,6332060696
3,first_24h_labs,8887920063


## Eligibility and label

Index admissions require age ≥18 by `anchor_age + year(admittime) - anchor_year`,
an unambiguous timeline, survival to discharge, a stay of at least 24 hours,
and no acute/psychiatric transfer destination. Timestamp and ID exclusions
are mutually exclusive at the source-row level. Valid short or planned
encounters still inform history and follow-up.

The first subsequent hospital encounter after discharge counts as positive
if it falls within (0, 30 days], is `DIRECT EMER.`, `EW EMER.` or `URGENT`,
and is neither a transfer nor otherwise ambiguous. A same-day return with a
positive gap can qualify. A zero gap, overlapping return, or intervening
planned, observation or transfer encounter makes the index ineligible for
the binary endpoint. A recorded death by day 30 also censors an index
without a qualifying return before or on the recorded death date.

A zero label denotes no qualifying event observed in MIMIC-IV. Short
observation, release-end censoring and care at outside hospitals can hide
returns. Date shifts preserve within-patient intervals. Insurance and
admission category fields are administrative values with no revision history
to independently reconstruct their exact state at the landmark.

## Laboratory and historical features

Use v3.1 Blood/Hematology for the three CBC analytes and Blood/Chemistry for
the six routine chemistry analytes. The specimen detail available in the
dictionary is Blood. CBC convention is whole blood and routine chemistry
convention is serum/plasma. The source does not document specimen subtype
for individual selected rows. Blood gas, point-of-care alternatives and
unlinked observations are outside this feature definition. The mapping is
verified against observed numeric counts and canonical units for this release.

Eligible measurements match both admission and patient. Chart and store
time must both fall by the prediction landmark, with storage no earlier
than charting. Select the earliest valid chart time, then earliest storage
time, then event ID and value. Missing/unknown units and nonnumeric or
nonfinite results are excluded. Unit conversions and upper clipping are not
applied. The private lab range report identifies extremes for review.

Count encounters started earlier and discharged at or before index admission.
Elapsed days since the most recent completed discharge are missing when
there is no prior completed encounter. No preprocessing is fitted here.

In [3]:
if RUN_COHORT and EXISTING_RUN is None:
    cohort, splits, flow, report, audit = run_workflow(
        client, location, plans, byte_cap=MAX_BYTES_BILLED
    )
    output = save_run(OUTPUT_DIR, schema, cohort, splits, flow, report, audit)
    print(f"Local cohort run saved to {output}")

## Descriptive evidence and patient splits

Review cohort flow, prevalence, lab availability, distributions by outcome
and the 70/15/15 patient split (seed 380). The notebook displays aggregate
evidence only. Blank counts can indicate small-cell suppression.
Apply the project’s privacy review before releasing any aggregate.

In [4]:
if output is not None:
    for table in ("cohort_summary", "cohort_flow", "admission_type", "age_summary",
                  "prior_admissions", "lab_mapping", "lab_missingness", "lab_completeness",
                  "labs_by_outcome", "partitions"):
        display(pd.read_csv(output / "aggregates" / f"{table}.csv"))
    for plot in ("cohort_flow", "label_distribution", "lab_missingness"):
        display(Image(filename=str(output / "aggregates" / f"{plot}.png")))